In [1]:
# ==============================================================================
# PROYECTO: Pinus-CO2 - Inteligencia de Captura de Carbono
# SCRIPT: 01_etl_procesamiento_co2.py
# DESCRIPCIÓN: Extracción, limpieza y feature engineering para cálculo de CO2
# ==============================================================================

# 1. IMPORTACIÓN DE LIBRERÍAS (Las herramientas de trabajo)
import pandas as pd
import numpy as np

print("Iniciando proceso ETL...")

Iniciando proceso ETL...


In [3]:
# ==============================================================================
# FASE 1: EXTRAER (Extract)
# ==============================================================================
# Leemos el archivo Excel original. 
# Nota: Asegúrate de que el archivo esté en la misma carpeta que tu script.
ruta_archivo = 'Datos CO2.xlsx'

try:
    # Leemos la hoja específica donde están los datos
    df_raw = pd.read_excel(ruta_archivo, sheet_name='PER')
    print(f"✅ Datos cargados exitosamente. Filas iniciales: {len(df_raw)}")
except FileNotFoundError:
    print("❌ Error: No se encontró el archivo. Revisa el nombre o la ruta.")

✅ Datos cargados exitosamente. Filas iniciales: 2955


In [4]:
# ==============================================================================
# FASE 2: TRANSFORMAR (Transform)
# ==============================================================================

# Hacemos una copia de seguridad para trabajar sobre ella
df_clean = df_raw.copy()

# 2.1 Limpieza de Datos (Data Cleaning)
# --------------------------------------------------
# Eliminamos filas duplicadas exactas 
df_clean = df_clean.drop_duplicates()

# Reemplazamos espacios en los nombres de columnas para evitar errores 
df_clean.columns = df_clean.columns.str.replace(' ', '_')

# Manejo de nulos (Tratamos la columna 'Ctronco' que tiene muchos NaN)
# Si es una cualidad del tronco que no usaremos para el volumen, la rellenamos con 'Sin Dato'
df_clean['Ctronco'] = df_clean['Ctronco'].fillna('Sin Dato')

# Filtramos posibles errores de medición (árboles sin volumen o sin altura)
df_clean = df_clean.dropna(subset=['Vol_T', 'HT_m', 'DAP_cm'])
df_clean = df_clean[df_clean['Vol_T'] > 0] # Nos aseguramos de que no haya volúmenes en cero o negativos

print(f"🧹 Limpieza completada. Filas después de limpieza: {len(df_clean)}")



🧹 Limpieza completada. Filas después de limpieza: 2927


In [6]:
# 2.2 Ingeniería de Características (Feature Engineering) - Fórmulas IPCC
# --------------------------------------------------
# Definimos los factores técnicos para pino (ajustables según la especie específica)
DENSIDAD_MADERA_KG_M3 = 500  # Densidad básica de la madera de pino
BEF = 1.3                    # Factor de Expansión de Biomasa (Incluye ramas y hojas)
FRACCION_CARBONO = 0.47      # Porcentaje de la biomasa que es carbono puro (IPCC estándar)
FACTOR_CO2 = 3.67            # Relación de peso molecular entre C y CO2 (44/12)

# Cálculo 1: Biomasa Aérea Total (en Kilogramos)
df_clean['Biomasa_Aerea_kg'] = df_clean['Vol_T'] * DENSIDAD_MADERA_KG_M3 * BEF

# Cálculo 2: Carbono Retenido (en Kilogramos)
df_clean['Carbono_Retenido_kg'] = df_clean['Biomasa_Aerea_kg'] * FRACCION_CARBONO

# Cálculo 3: Dióxido de Carbono Equivalente (CO2eq) en Kilogramos y Toneladas
df_clean['CO2_Eq_kg'] = df_clean['Carbono_Retenido_kg'] * FACTOR_CO2
df_clean['CO2_Eq_Ton'] = df_clean['CO2_Eq_kg'] / 1000

# Añadimos una columna calculada de Edad del árbol (asumiendo medición en 2026)
año_medicion = 2026
df_clean['Edad_Años'] = año_medicion - df_clean['Año_de_siembra']

print(" Cálculos alométricos y de captura de carbono aplicados.")

 Cálculos alométricos y de captura de carbono aplicados.


In [7]:
# ==============================================================================
# FASE 3: CARGAR (Load)
# ==============================================================================

# Seleccionamos y ordenamos las columnas que realmente le importan al negocio 
columnas_finales = [
    'ID_Parcela', 
    'Predio', 
    'Año_de_siembra',
    'Edad_Años',
    'INDV', 
    'DAP_cm', 
    'HT_m', 
    'A_Basal_(m2)', 
    'Vol_T', 
    'Biomasa_Aerea_kg',
    'CO2_Eq_Ton'
]

df_final = df_clean[columnas_finales]

# Exportamos el resultado limpio a un archivo CSV 
nombre_archivo_salida = 'Dataset_Carbono_PowerBI.csv'
df_final.to_csv(nombre_archivo_salida, index=False, encoding='utf-8-sig')

print(f" ¡ETL Finalizado con éxito! El archivo limpio se guardó como: {nombre_archivo_salida}")
print(df_final[['ID_Parcela', 'Predio', 'Edad_Años', 'Vol_T', 'CO2_Eq_Ton']].head())

 ¡ETL Finalizado con éxito! El archivo limpio se guardó como: Dataset_Carbono_PowerBI.csv
  ID_Parcela      Predio  Edad_Años     Vol_T  CO2_Eq_Ton
0      PL129  Pericuara          10  0.016061    0.018007
2      PL169  Pericuara          10  0.031864    0.035726
3      PL154  Pericuara          10  0.032475    0.036410
4        PO6  Pericuara          10  0.011960    0.013410
5        PL9  Pericuara          10  0.023759    0.026639
